# 模組 2：從平均數到神經網路——搭配練習筆記本

這份筆記本是「從平均數到神經網路」模組的實作搭配教材。
它依序收錄了**模組中的每一段程式碼範例**，並針對剛接觸 Python 資料科學工具鏈的
程式設計師加入額外說明。如果你已經知道怎麼 `pip install` 套件、也會寫 `for` 迴圈，
可以快速瀏覽框起來的說明文字，直接看程式碼即可。

**使用方式：** 請由上而下依序執行每個儲存格。後面的段落會沿用前面段落定義的變數
（例如 `x`、`y`、`rng` 等），就跟原始模組範例彼此銜接的方式一樣。如果你跳著看
遇到 `NameError`，往上捲動並重新執行定義該變數的儲存格即可。

---

## 環境設定

在執行任何程式碼之前，你需要先安裝 Python 本身（任何近期的 Python 3.9+ 版本都可以）
以及幾個第三方套件。「第三方」的意思是這些套件並非 Python 標準函式庫的一部分——
它們被發佈在 **Python Package Index（PyPI）**，也就是可安裝 Python 套件的中央倉庫，
而你會使用 Python 內建的套件管理工具 **pip** 來安裝它們。

打開終端機並執行：

```bash
pip install numpy scipy scikit-learn matplotlib pandas jupyter
```

以下簡單說明每個套件的用途（在這份筆記本中你會一一正式認識它們）：

- **numpy** — 快速的陣列運算與線性代數。其他一切都建立在它之上。
- **scipy** — 建立在 numpy 之上的額外科學／統計運算函式。
- **scikit-learn**（匯入時寫作 `sklearn`）— 現成可用的機器學習模型與工具。
- **matplotlib** — 繪圖與圖表。
- **pandas** — 有標籤的表格式資料（可以想成「Python 裡的試算表」）。
- **jupyter** — 筆記本環境本身（也就是你現在正在閱讀的東西）。

如果你是在既有的 Jupyter 筆記本中執行，而發現缺少某個套件，也可以直接在儲存格內
安裝，只要在指令前面加上 `!` 即可，例如 `!pip install scipy`。這樣會讓指令在你的
shell 中執行，而不是當作 Python 程式碼執行。

再補充一個風格上的說明：在整份筆記本中你會看到 `import numpy as np`。`as np`
的部分只是一個暱稱——它讓你可以打 `np.mean(...)`，而不用打比較長的
`numpy.mean(...)`。這些縮寫（`np` 代表 numpy、`pd` 代表 pandas、`plt` 代表
`matplotlib.pyplot`）在 Python 資料科學圈裡已經是通用慣例，如果用別的名稱反而
會讓其他讀你程式碼的人感到困惑。


## 第 1 節 —— 描述資料

### 平均數與變異數

我們第一個要匯入的是 `numpy`，幾乎總是暱稱為 `np`。numpy 最主要的特色是
**陣列（array）**——可以把它想成一種專門為快速、整批運算而生的 Python 清單（list）。
`np.array([...])` 會把一般的 Python 清單轉換成 numpy 陣列；從那之後，
像 `np.mean()` 和 `np.var()` 這類函式就可以一次對整個陣列做運算，完全不需要手動寫迴圈。

- `np.mean(x)` — `x` 中所有數值的平均。
- `np.var(x)` — 母體變異數（population variance，請參考本儲存格之後的說明，
  裡面提到 `ddof` 參數——這點很重要）。
- `np.sqrt(...)` — 開平方根；這裡用來把變異數換算成標準差。

`f"..."` 字串（稱為 f-string）可以讓你直接把變數放進文字裡，例如
`f"mean = {mean_x:.2f}"` 會印出 `mean_x` 並四捨五入到小數點後兩位。


In [ ]:
import numpy as np

x = np.array([12, 15, 14, 10, 18, 22, 9, 17])

mean_x = np.mean(x)
var_x = np.var(x)  # population variance (divides by n)

print(f"mean = {mean_x:.2f}")
print(f"variance = {var_x:.2f}")
print(f"standard deviation = {np.sqrt(var_x):.2f}")


**有一個小地方值得先知道：** `np.var()` 預設是除以 `n`（也就是「母體」變異數）。
許多統計教科書、以及 pandas，則是改成除以 `n - 1`（「樣本」變異數，這是針對
「你用來估計平均數的資料，跟你拿來測量離散程度的資料是同一批」這件事所做的
一個小修正）。如果你想要那個修正，可以在 `np.var()` 中加上 `ddof=1`：
`np.var(x, ddof=1)`。這不會改變本模組中的任何結論，但如果沒有對齊設定，
算出來的數字會跟 pandas 的 `.var()` 有些微差異——這點值得記住，免得在實作練習時
浪費時間排查。

### 共變異數（Covariance）

`np.cov()` 計算兩個變數共同變動的程度。它回傳的是完整的 2x2 矩陣，而不是單一數字——
用 `[0, 1]` 做索引才能取出我們真正需要的那個數字。


In [ ]:
x = np.array([1, 2, 3, 4, 5, 6])
y = np.array([2, 4, 5, 4, 5, 8])

cov_xy = np.cov(x, y, ddof=0)[0, 1]
print(f"covariance(x, y) = {cov_xy:.2f}")


`np.cov(x, y, ddof=0)` 會回傳：

```
[[ var(x)       cov(x, y) ]
 [ cov(x, y)    var(y)    ]]
```

對角線上是各變數自身的變異數；非對角線的部分（兩側相等）則是它們之間的共變異數。
`ddof=0` 是告訴 numpy 使用母體版本（除以 `n`），這與前一個儲存格中
`np.var()` 的預設值一致。

### 相關係數（Correlation）

`np.corrcoef()` 的運作方式類似——它同樣回傳一個矩陣，用 `[0, 1]` 取出
`x` 與 `y` 之間的相關係數，這個數字永遠介於 -1 到 1 之間。


In [ ]:
corr_xy = np.corrcoef(x, y)[0, 1]
print(f"correlation(x, y) = {corr_xy:.3f}")


### 觀察資料本身，而不只是摘要數字

`pandas`（暱稱 `pd`）是 Python 中處理表格式資料的標準函式庫——可以把它的核心物件
`DataFrame` 想成是活在你程式碼裡的一份試算表。這是我們第一次使用它。
`pd.DataFrame({...})` 會用一個字典建立一張表格，字典的每個鍵（key）會變成欄位名稱，
每個值則是該欄位的資料。接著 `.describe()` 會一次印出每個數值欄位的筆數、平均數、
標準差、最小值、最大值以及四分位數——是快速初步檢視任何新資料集的好方法。


In [ ]:
import pandas as pd

df = pd.DataFrame({"x": x, "y": y})
print(df.describe())


### 資料標準化（z 分數）

減去平均數，再除以標準差。結果永遠是平均數為 0、標準差為 1，這使得原本尺度
完全不同的變數（例如金額和年數）也能拿來互相比較。


In [ ]:
z = (x - np.mean(x)) / np.std(x)
print(f"standardized mean = {np.mean(z):.4f}, standardized std = {np.std(z):.4f}")


### 離群值與穩健性

`np.append(x, 500)` 會回傳一個在尾端加上 `500` 的新陣列（numpy 陣列的大小是固定的，
所以「附加」（append）永遠是產生一個新陣列，而不是就地擴大原本的陣列——
這跟 Python 清單有個微妙但重要的差異）。


In [ ]:
x_with_outlier = np.append(x, 500)   # one extreme value added
print(f"mean without outlier: {np.mean(x):.2f}, with outlier: {np.mean(x_with_outlier):.2f}")
print(f"std without outlier: {np.std(x):.2f}, with outlier: {np.std(x_with_outlier):.2f}")


### 一次描述多個變數

這裡再介紹一個新工具：`np.random.default_rng(seed)` 會建立一個**隨機數產生器
（random number generator）**物件，通常稱為 `rng`。傳入固定的 `seed`（任意整數）
表示每次呼叫 `rng.uniform(...)` 或 `rng.normal(...)` 時，都會產生*完全相同*的
「隨機」數字——這正是讓範例可重現的關鍵。模組會在第 2 節正式介紹這個概念；
我們在這裡提早建立 `rng`，是為了讓這個儲存格能夠獨立執行。

- `rng.uniform(low, high, size)` 會抽取 `size` 個介於 `low` 與 `high` 之間、
  均勻分布的數字。
- `rng.normal(mean, std, size)` 會從常態（鐘形曲線）分布中抽取 `size` 個數字。
- `df.corr()` 一次計算 DataFrame 中每一對欄位之間的相關係數——也就是相關係數矩陣。


In [ ]:
import pandas as pd

rng = np.random.default_rng(7)  # created here so this example is self-contained

df = pd.DataFrame({
    "hours_studied": rng.uniform(0, 10, 50),
    "sleep_hours": rng.uniform(4, 9, 50),
})
df["exam_score"] = 5 * df["hours_studied"] + 2 * df["sleep_hours"] + rng.normal(0, 5, 50)

print(df.corr())


## 第 2 節 —— 擬合直線

我們在這裡用固定的種子（`42`）重新建立 `rng`，讓這一節的範例不受第 1 節
上面產生的隨機數影響，仍能獨立重現。

- `rng.uniform(0, 10, n)` — 產生 `n` 個介於 0 到 10 之間的隨機 x 值。
- `rng.normal(0, 4.0, n)` — 產生 `n` 個隨機雜訊值，加到一個「真實的」線性關係上，
  讓資料看起來像是帶有雜訊的真實測量值，而不是一條完美的直線。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, n)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, n)   # true slope 2.5, true intercept 3.0

beta1_hat = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
beta0_hat = np.mean(y) - beta1_hat * np.mean(x)

print(f"fitted slope (beta1) = {beta1_hat:.3f}")
print(f"fitted intercept (beta0) = {beta0_hat:.3f}")


### 用函式庫核對手算的公式

`scipy` 是建立在 numpy 之上的科學計算函式庫。這裡我們只需要用到其中一個函式：
`scipy.stats.linregress`，它會幫你擬合一條直線，並回傳一個帶有 `.slope`（斜率）
和 `.intercept`（截距）屬性的物件（此外還有其他資訊，例如相關係數）。
如果尚未安裝 scipy：`pip install scipy`。


In [ ]:
from scipy import stats

result = stats.linregress(x, y)
print(f"scipy slope = {result.slope:.3f}, intercept = {result.intercept:.3f}")


### 矩陣形式的解法

`np.column_stack([...])` 會把幾個一維陣列黏合成單一二維陣列的並排欄位——這裡是把
一欄全為 `1` 的陣列與 `x` 這一欄接在一起，形成「設計矩陣（design matrix）」`X`。
`X.T` 是轉置（把列與欄互換），`@` 是矩陣乘法（numpy 專門用於矩陣乘法的運算子，
和逐元素相乘的 `*` 不同），而 `np.linalg.inv(...)` 則是計算矩陣的反矩陣。
把這些組合起來，`np.linalg.inv(X.T @ X) @ X.T @ y` 就是直接求解「正規方程式
（normal equations）」的公式。


In [ ]:
X = np.column_stack([np.ones(n), x])   # design matrix: intercept column + x column
beta_matrix = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"matrix-form solution: {beta_matrix}")


### 多元迴歸：只要再加一欄

若要用兩個輸入變數（`x` 和 `x2`）而不是一個來預測 `y`，不需要任何新的運算機制——
只需要在 `X` 中多加一欄即可。


In [ ]:
x2 = 0.5 * x + rng.normal(0, 2, n)
y_multi = 2.5 * x - 1.2 * x2 + 3.0 + rng.normal(0, 4.0, n)

X_multi = np.column_stack([np.ones(n), x, x2])
beta_multi = np.linalg.inv(X_multi.T @ X_multi) @ X_multi.T @ y_multi
print(f"multiple regression coefficients: {beta_multi}")


### 用 scikit-learn 做同樣的事

`scikit-learn`（匯入時寫作 `sklearn`）是 Python 中標準的機器學習函式庫。
它的所有模型都共用同一套介面：建立模型物件、對你的資料呼叫 `.fit(X, y)`，
然後讀取結果（這裡是 `.coef_` 和 `.intercept_`）。請注意，scikit-learn
會自動加上自己的截距欄，所以我們只傳入 `x` 和 `x2` 這兩欄
（`X_multi[:, 1:]` —— 所有列，但*除了*第一欄以外的所有欄）。
如果尚未安裝 sklearn：`pip install scikit-learn`。


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(X_multi[:, 1:], y_multi)  # sklearn adds its own intercept
print(f"sklearn coefficients: {model.coef_}, intercept: {model.intercept_}")


### 曲線仍然是「線性」模型

加上一欄 `x**2`，同一套運算機制就能擬合出一條拋物線。「線性」指的是在*參數*上
是線性的，而不是指最終曲線的形狀。


In [ ]:
X_quadratic = np.column_stack([np.ones(n), x, x**2])
beta_quadratic = np.linalg.inv(X_quadratic.T @ X_quadratic) @ X_quadratic.T @ y
print(f"quadratic fit coefficients: {beta_quadratic}")


### 一個完全手算的小型範例

只有五個資料點，這樣你可以看清楚牽涉到的每一個數字，而不會被大量輸出淹沒。


In [ ]:
x_small = np.array([1, 2, 3, 4, 5])
y_small = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

beta1 = np.cov(x_small, y_small, ddof=0)[0, 1] / np.var(x_small)
beta0 = np.mean(y_small) - beta1 * np.mean(x_small)
print(f"fitted line: y = {beta0:.2f} + {beta1:.2f} * x")

for xi, yi in zip(x_small, y_small):
    prediction = beta0 + beta1 * xi
    print(f"x={xi}: actual y={yi:.1f}, predicted y={prediction:.2f}, residual={yi - prediction:.2f}")


## 第 3 節 —— 衡量誤差

我們重新建立與第 2 節相同的擬合直線（同樣的種子、同樣的資料），讓這一節可以
獨立成立，接著計算它的殘差（residuals）——也就是每一點上剩餘的誤差，
即「實際值 - 預測值」。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, n)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, n)

beta1_hat = np.cov(x, y, ddof=0)[0, 1] / np.var(x)
beta0_hat = np.mean(y) - beta1_hat * np.mean(x)
y_hat = beta0_hat + beta1_hat * x

residuals = y - y_hat
print(f"mean residual = {np.mean(residuals):.4f}")   # should be ~0 for OLS, always
print(f"residual std dev = {np.std(residuals):.3f}")


### 均方誤差（MSE）與 R 平方

`sklearn.metrics` 是 scikit-learn 提供的一組現成評分函式集合——不需要自己手寫
平均／平方的運算邏輯。


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y, y_hat)
print(f"MSE = {mse:.3f}")


In [ ]:
r2 = r2_score(y, y_hat)
print(f"R^2 = {r2:.3f}")


### 平均絕對誤差（MAE）

概念和 MSE 相同，但這裡是取每個殘差絕對值的平均，而不是取平方——
所以一個極大的離群誤差不會像在 MSE 中那樣主導整個分數。


In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y, y_hat)
print(f"MAE = {mae:.3f}  (compare to MSE = {mse:.3f} and RMSE = {np.sqrt(mse):.3f})")


### 並排比較兩個候選模型

`x.reshape(-1, 1)` 把一維陣列轉換成只有一欄的二維陣列——scikit-learn 的模型
永遠期望輸入的 `X` 是二維的（列 = 樣本，欄 = 特徵），即使只有一個特徵也一樣。
`-1` 是告訴 numpy「自動算出這個維度的大小」。


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X_linear = x.reshape(-1, 1)
X_quad = np.column_stack([x, x**2])

for name, X_features in [("linear", X_linear), ("quadratic", X_quad)]:
    m = LinearRegression().fit(X_features, y)
    preds = m.predict(X_features)
    print(
        f"{name:10s}  MSE={mean_squared_error(y, preds):7.2f}  "
        f"MAE={mean_absolute_error(y, preds):6.2f}  "
        f"R2={r2_score(y, preds):.3f}"
    )


## 第 4 節 —— 最佳化參數

### 訓練／測試集分割，以及初探過度擬合

一批新的匯入，全都來自 scikit-learn：

- `train_test_split` — 隨機把你的資料切分成訓練集和保留的測試集（這裡是
  70% / 30%，透過 `test_size=0.3` 設定）。`random_state=0` 是 scikit-learn
  版本的隨機種子——固定它，切分結果就能重現。
- `PolynomialFeatures(degree)` — 自動幫你建立 `x`、`x**2`、`x**3`……這些欄位，
  你不用自己手動一個個打出來。
- `make_pipeline(...)` — 把前處理步驟和模型串接成單一物件，讓 `.fit()` 和
  `.predict()` 可以一次呼叫就跑完整條流程。


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(1)
n = 40
x = np.linspace(-3, 3, n)
y = 0.5 * x**3 - 2 * x**2 + x + 5 + rng.normal(0, 6, n)

X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1, 1), y, test_size=0.3, random_state=0
)

for degree in [1, 3, 11]:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, model.predict(X_train))
    test_mse = mean_squared_error(y_test, model.predict(X_test))
    print(f"degree {degree:2d}: train MSE = {train_mse:7.2f}   test MSE = {test_mse:7.2f}")


`np.linspace(start, stop, n)` —— 和 `rng.uniform` 不同，這個函式會產生 `n` 個
從 `start` 到 `stop`*等間距*的數值（不是隨機的），很適合用來畫平滑的曲線。

### 從零開始寫梯度下降

這裡沒有任何新的匯入——一切都是用你已經用過的 numpy 運算組合而成。下面的函式
直接實作了梯度下降的運算機制：從一個猜測值開始（`w = np.zeros(...)`，全為 0），
反覆計算平方誤差損失函數的梯度，然後朝反方向邁出一小步。


In [ ]:
def gradient_descent_linreg(X, y, lr=0.1, n_iter=200):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    loss_history = []
    for _ in range(n_iter):
        y_pred = X @ w
        grad = -(2 / n_samples) * X.T @ (y - y_pred)
        w = w - lr * grad
        loss_history.append(np.mean((y - y_pred) ** 2))
    return w, loss_history

x_data = rng.uniform(0, 10, 60)
y_data = 2.5 * x_data + 3.0 + rng.normal(0, 4.0, 60)

x_std = (x_data - x_data.mean()) / x_data.std()   # standardizing helps convergence
X = np.column_stack([np.ones(60), x_std])

w, loss_history = gradient_descent_linreg(X, y_data, lr=0.3, n_iter=200)
print(f"final weights: {w}")
print(f"loss after 1 iteration: {loss_history[0]:.2f}, after 200: {loss_history[-1]:.2f}")


### 交叉驗證（Cross-validation）

`cross_val_score` 會自動完成「切成 k 折、訓練／測試 k 次、平均各次分數」的流程。
`scoring="neg_mean_squared_error"` 之所以是*負的* MSE，是因為 scikit-learn 的慣例
是分數越高代表越好——我們把它取負號（`-scores`）換回一般的正值 MSE 來印出。


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

scores = cross_val_score(
    LinearRegression(), x.reshape(-1, 1), y, cv=5, scoring="neg_mean_squared_error"
)
print(f"per-fold MSE: {-scores}")
print(f"average MSE across folds: {-scores.mean():.3f}")


## 第 5 節 —— 從直線延伸到多維模型

### 用數字網格視覺化損失曲面

這裡沒有新的匯入——只是再次使用 numpy，用來建立一個涵蓋各種可能 `(b0, b1)`
組合的損失值網格。`[[... for b0 in b0_grid] for b1 in b1_grid]` 是一個
**巢狀串列生成式（nested list comprehension）**：對每一個 `b1`，計算每一個
`b0` 對應的損失，產生一個二維網格，這個網格可以用 matplotlib 的 3D 繪圖功能
畫成曲面（為了讓這個儲存格執行快一點，這裡沒有實際畫出來）。


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n = 60
x = rng.uniform(0, 10, n)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, n)

def loss_surface(b0, b1, x, y):
    return np.mean((y - (b0 + b1 * x)) ** 2)

b0_grid = np.linspace(-15, 20, 60)
b1_grid = np.linspace(-2, 6, 60)
loss_values = np.array([[loss_surface(b0, b1, x, y) for b0 in b0_grid] for b1 in b1_grid])

print(f"loss surface shape: {loss_values.shape}")
print(f"minimum loss on this grid: {loss_values.min():.2f}")


### 非凸曲面：Himmelblau 函數

這次全是純 Python 函式（不需要 sklearn 或 scipy）——`himmelblau` 是這個地形本身，
`himmelblau_grad` 是它的梯度（用微積分手動推導出來），而 `gradient_descent`
則是第 4 節那個最佳化器的簡化通用版本。


In [ ]:
def himmelblau(x, y):
    return (x**2 + y - 11) ** 2 + (x + y**2 - 7) ** 2

def himmelblau_grad(p):
    x, y = p
    dfdx = 4 * x * (x**2 + y - 11) + 2 * (x + y**2 - 7)
    dfdy = 2 * (x**2 + y - 11) + 4 * y * (x + y**2 - 7)
    return np.array([dfdx, dfdy])

def gradient_descent(grad_fn, start, lr=0.01, n_iter=300):
    p = np.array(start, dtype=float)
    for _ in range(n_iter):
        p = p - lr * grad_fn(p)
    return p

for start in [(-4, 4), (4, 4), (-4, -4), (4, -2)]:
    end_point = gradient_descent(himmelblau_grad, start)
    print(f"started at {start}, converged near {np.round(end_point, 2)}")


### 一個小實驗：隨著維度增加，局部最小值會變得越來越少見

`np.linalg.eigvalsh(...)` 計算對稱矩陣的特徵值——這裡用它來檢驗一個隨機點是否
在每個方向上「同時」向上彎曲（所有特徵值都是正的，代表是一個真正的局部最小值）。


In [ ]:
rng = np.random.default_rng(0)

def fraction_true_local_minima(dimension, n_trials=300):
    count = 0
    for _ in range(n_trials):
        A = rng.normal(size=(dimension, dimension))
        hessian_stand_in = (A + A.T) / 2  # a random symmetric matrix, standing in for a Hessian
        eigenvalues = np.linalg.eigvalsh(hessian_stand_in)
        if np.all(eigenvalues > 0):        # positive in every direction => true local min
            count += 1
    return count / n_trials

for d in [2, 4, 8, 12]:
    frac = fraction_true_local_minima(d)
    print(f"dimension {d:2d}: fraction of critical points that are true local minima = {frac:.3f}")


### 用 `StandardScaler` 做特徵縮放

`StandardScaler` 做的正是第 1 節那種 z 分數標準化（`(x - mean) / std`），
只不過它是一個可重複使用、帶有 `.fit_transform()` 方法的 scikit-learn 物件——
當你有很多欄需要一次標準化時特別方便，不用一欄一欄處理。


In [ ]:
from sklearn.preprocessing import StandardScaler

X_raw = np.column_stack([x_data, x_data * 1000])  # two features on very different scales
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
print(f"raw feature ranges: {X_raw.min(axis=0)} to {X_raw.max(axis=0)}")
print(f"scaled feature means: {X_scaled.mean(axis=0).round(3)}, stds: {X_scaled.std(axis=0).round(3)}")


## 第 6 節 —— 從機器學習到人工智慧

### 從零開始用 numpy 建立一個小型神經網路

沒有新的函式庫——神經網路的前向傳播（forward pass）其實就只是矩陣乘法
（`@`）加上一個簡單的非線性函數（`relu`，作用只是把負數歸零），
夾在兩層權重之間應用而已。


In [ ]:
import numpy as np

def relu(z):
    return np.maximum(0, z)

def simple_forward_pass(x, W1, b1, W2, b2):
    '''
    x:  input vector
    W1, b1: weights and bias of the first (hidden) layer
    W2, b2: weights and bias of the second (output) layer
    '''
    hidden = relu(W1 @ x + b1)      # first layer: weighted sum, then nonlinearity
    output = W2 @ hidden + b2       # second layer: another weighted sum
    return output

rng = np.random.default_rng(0)
x_input = rng.normal(size=3)         # 3 input features
W1 = rng.normal(size=(4, 3)) * 0.5   # hidden layer: 4 hidden units
b1 = np.zeros(4)
W2 = rng.normal(size=(1, 4)) * 0.5   # output layer: 1 output value
b2 = np.zeros(1)

prediction = simple_forward_pass(x_input, W1, b1, W2, b2)
print(f"network output: {prediction}")


### 讓 scikit-learn 訓練一個真正的（小型）神經網路

`MLPRegressor`（「多層感知器」，Multi-Layer Perceptron）是 scikit-learn 內建的
神經網路模型。它遵循和 `LinearRegression` 一模一樣的 `.fit()` / `.predict()`
介面——這種一致性是 scikit-learn 刻意設計的。

- `hidden_layer_sizes=(16, 16)` — 兩層隱藏層，每層 16 個神經元。
- `activation="relu"` — 各層之間套用的非線性函數。
- `max_iter=2000` — 放棄之前允許的最大訓練迭代次數。


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

x_data = np.linspace(-3, 3, 100).reshape(-1, 1)
y_data = np.sin(x_data).ravel() + rng.normal(0, 0.1, 100)

net = MLPRegressor(
    hidden_layer_sizes=(16, 16),   # two hidden layers, 16 units each
    activation="relu",
    max_iter=2000,
    random_state=0,
)
net.fit(x_data, y_data)
predictions = net.predict(x_data)

print(f"training MSE: {mean_squared_error(y_data, predictions):.4f}")
print(f"number of weight matrices: {len(net.coefs_)}")
print(f"total trainable parameters: {sum(w.size for w in net.coefs_) + sum(b.size for b in net.intercepts_)}")


（`.ravel()` 會把只有一欄的二維陣列攤平回一般的一維陣列——這裡之所以需要它，
是因為 `x_data` 為了符合 scikit-learn 的要求而是二維的，但 `np.sin(x_data)`
會繼承同樣的二維形狀，而我們需要的是攤平後的一維目標值陣列。）


## 第 7 節 —— 用 Python 執行這一切

這一節是整個工具鏈的導覽。如果你還沒安裝所有套件，一個指令就能涵蓋整個模組：

```bash
pip install numpy scipy scikit-learn matplotlib pandas
```

### NumPy：從頭到尾的陣列與線性代數

第 2 節完整的正規方程式計算，整合在這裡。


In [ ]:
import numpy as np

# The normal equations, end to end, using only NumPy
rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 100)
y = 2.5 * x + 3.0 + rng.normal(0, 4.0, 100)

X = np.column_stack([np.ones_like(x), x])
beta = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"beta = {beta}")


`np.ones_like(x)` 會建立一個和 `x` 形狀相同、全為 `1` 的陣列——比起手動寫
`np.ones(len(x))`，這是一個小小的便利寫法。

### SciPy：獨立核對


In [ ]:
from scipy import stats

result = stats.linregress(x, y)
print(f"slope = {result.slope:.3f}, intercept = {result.intercept:.3f}, r-squared = {result.rvalue**2:.3f}")


### scikit-learn：先 fit，再 predict


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    x.reshape(-1, 1), y, test_size=0.3, random_state=0
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print(f"test MSE = {mean_squared_error(y_test, predictions):.3f}")
print(f"test R^2 = {r2_score(y_test, predictions):.3f}")


### Matplotlib：看見數字所代表的意義

`import matplotlib.pyplot as plt` 是 matplotlib 主要繪圖介面的標準暱稱——`plt`。
`plt.scatter` 畫出散佈的點，`plt.plot` 畫出連續的線，而 `plt.show()`
會顯示完成的圖形（在 Jupyter 筆記本中，即使不呼叫 `.show()`，圖形通常也會
自動顯示，但養成加上它的習慣仍是好做法）。


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x, y, alpha=0.7)
plt.plot(sorted(x), model.predict(np.sort(x).reshape(-1, 1)), color="crimson")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data with fitted line")
plt.show()


### pandas：帶有標籤的表格式資料

我們在這裡重複使用第 6 節神經網路範例中的 `x_data` / `y_data`，來示範如何
在 DataFrame 和 scikit-learn 所需的一般 numpy 陣列之間搬移資料。


In [ ]:
import pandas as pd

df = pd.DataFrame({"x": x_data.ravel(), "y": y_data})
X_from_df = df[["x"]].to_numpy()   # extract as a NumPy array for sklearn
y_from_df = df["y"].to_numpy()

model = LinearRegression().fit(X_from_df, y_from_df)
print(f"fitted from a DataFrame: slope = {model.coef_[0]:.3f}")


### 可重現性（Reproducibility）

整份筆記本都仰賴這個模式：固定一個種子，「隨機」數字就會變成完全可以重現的。


In [ ]:
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(7)
print(np.array_equal(rng_a.normal(size=5), rng_b.normal(size=5)))  # True: identical seeds, identical draws


### 快速對照表：概念 → 函式庫

| 概念 | 章節 | 函式庫／函式 |
|---|---|---|
| 平均數、變異數、共變異數、相關係數 | 1 | `numpy` |
| 多變數之間的相關係數矩陣 | 1 | `pandas.DataFrame.corr()` |
| 正規方程式（手算） | 2 | `numpy.linalg.inv` |
| 最小平方擬合（驗證用） | 2 | `scipy.stats.linregress` |
| 多元／多項式迴歸 | 2 | `sklearn.linear_model.LinearRegression`、`PolynomialFeatures` |
| MSE、MAE、R² | 3 | `sklearn.metrics` |
| 訓練／測試集分割、交叉驗證 | 4 | `sklearn.model_selection` |
| 梯度下降（從零開始） | 4 | `numpy` |
| 損失曲面、非凸最佳化 | 5 | `numpy` + `matplotlib`（3D 繪圖） |
| 特徵縮放 | 5 | `sklearn.preprocessing.StandardScaler` |
| 所有圖表與圖形 | 1–5 | `matplotlib` |


## 第 8 節 —— 從擬合直線到在照片中辨認一隻貓

### 步驟 1 —— 一張照片其實就是一張很大的數字表格

`sklearn.datasets.load_digits()` 提供了一個小型、內建於 scikit-learn 中的
手寫數字圖片資料集——不需要下載、也不需要網路連線，這正是我們在這裡使用它、
而不是真正的貓咪照片的原因。每張圖片是一個 8x8 的數字網格（像素亮度值），
以 numpy 陣列的形式儲存。


In [ ]:
import numpy as np
from sklearn.datasets import load_digits

digits = load_digits()
image = digits.images[0]         # an 8x8 grayscale image, as a NumPy array
print(image.shape)                 # (8, 8)
print(image)                        # each entry is a pixel brightness value


### 步驟 2 —— 把每個像素當作一個預測變數，並把分類問題當作擬合問題

`image.reshape(len(images), -1)` 會把每張 8x8 的網格**攤平**成一列 64 個數字——
概念和攤平表格一樣，只是這裡是對資料集中的每一張圖片同時套用。
`digits.target` 存放每張圖片對應的真實數字（0-9）——也就是我們想要預測的標籤。

`LogisticRegression` 是 scikit-learn 用於分類（預測一個類別，而不是一個數值）
的模型——採用的是你已經用過好幾次的同一套 `.fit()` / `.predict()` 介面。
`accuracy_score` 則回報模型在測試圖片中分類正確的比例。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = digits.images.reshape(len(digits.images), -1)   # flatten each 8x8 image to 64 numbers
y = digits.target                                     # the true digit, 0-9, for each image

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

print(f"test accuracy: {accuracy_score(y_test, predictions):.3f}")
print(f"number of weights being fit: {clf.coef_.size}")


### 步驟 4 —— 手工設計一個卷積核（convolutional kernel）

這裡沒有新的函式庫——這是一個從零開始、純用 numpy 實作的單一 2D 卷積運算，
目的是在你接觸真正深度學習函式庫（速度快得多、由 GPU 加速）中的同一種運算
之前，先讓這個運算機制變得具體可見。


In [ ]:
def apply_kernel(image, kernel):
    '''A minimal, from-scratch 2D convolution -- no padding, unit stride.'''
    kh, kw = kernel.shape
    ih, iw = image.shape
    out_h, out_w = ih - kh + 1, iw - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i + kh, j:j + kw]
            output[i, j] = np.sum(patch * kernel)   # the same weighted-sum idea, applied locally
    return output

vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
])

sample_image = digits.images[0]
edges = apply_kernel(sample_image, vertical_edge_kernel)
print(f"original shape: {sample_image.shape}, after kernel: {edges.shape}")
print(np.round(edges, 1))


---

## 你已經走過了整個技術堆疊

這份筆記本中的每一個儲存格都建立在前面的基礎之上：`numpy` 陣列存放數字，
`scipy` 和 `sklearn` 在這些數字上擬合並評估模型，`pandas` 把它們整理成表格，
而 `matplotlib` 則讓你能夠觀察結果。這個組合——以及擬合模型、衡量誤差、
並在模型未見過的資料上驗證的這個習慣——正是從一個雙參數迴歸到一個完整
神經網路，背後貫穿始終的同一個循環。
